# 01 — Data Loading

Загрузка данных из Excel в DB.

**Pipeline:**
1. **ExcelLoader** — загрузка IS/BS/CF + canonical sheets (debt, PPE, macro, segments)
2. **UnifiedExcelLoader** — загрузка ВСЕХ листов (history + schedules + corkscrews)
3. **Verification** — проверка данных в DB

Рекомендуется использовать **Этап 2** (unified loader) как основной метод загрузки.
Он покрывает все 31 лист шаблона, включая schedule-расписания.

In [ ]:
# ── Setup ──────────────────────────────────────────────
import sys, pathlib
ROOT = pathlib.Path('.').resolve()
for _p in [ROOT] + list(ROOT.parents):
    if (_p / 'engine').is_dir():
        ROOT = _p; break
sys.path.insert(0, str(ROOT))

COMPANY = None
for _p in [pathlib.Path('.').resolve()] + list(pathlib.Path('.').resolve().parents):
    if _p.parent.name == 'companies':
        COMPANY = _p.name; break
if not COMPANY:
    raise RuntimeError('Cannot detect company_id — run this notebook from companies/<id>/notebooks/')

print(f'Company: {COMPANY}')
print(f'ROOT:    {ROOT}')

In [ ]:
# ── Find Excel file ────────────────────────────────────
from pathlib import Path

excel_dir = ROOT / 'companies' / COMPANY / 'data' / 'excel'
candidates = [
    excel_dir / f'{COMPANY}_unified.xlsx',
    excel_dir / f'{COMPANY}_input.xlsx',
]
# Fallback: any xlsx in data/excel/
excel_path = None
for c in candidates:
    if c.exists():
        excel_path = c
        break
if not excel_path:
    xlsx_files = sorted(excel_dir.glob('*.xlsx')) if excel_dir.exists() else []
    if xlsx_files:
        excel_path = xlsx_files[0]

if not excel_path or not excel_path.exists():
    raise FileNotFoundError(f'No Excel file found in {excel_dir}')

print(f'Excel: {excel_path.name}')

# Show available sheets
import openpyxl
wb = openpyxl.load_workbook(excel_path, read_only=True)
print(f'Sheets ({len(wb.sheetnames)}): {wb.sheetnames}')
wb.close()

## Этап 1: ExcelLoader (IS/BS/CF + canonical)

Загружает core financial statements и canonical sheets (debt, PPE, macro, segments).
Использует маппинг из `configs/excel_loader.yaml`.

In [ ]:
from engine.loader.excel import ExcelLoader
from engine.database.repository import Repository
from engine import DB_PATH

with Repository(DB_PATH) as repo:
    loader = ExcelLoader(
        company_id=COMPANY,
        repo=repo,
        db_unit='USD',
        input_default_unit='mUSD',
    )
    result = loader.load(excel_path)

print(f'Rows written: {result.rows_written}')
if result.errors:
    print(f'ERRORS: {result.errors}')
if result.warnings:
    print(f'Warnings: {result.warnings[:5]}')
if result.missing_required:
    print(f'Missing required: {result.missing_required}')

## Этап 2: Unified Loader (все 31 лист)

Загружает ВСЕ данные из unified Excel:
- History IS/BS/CF
- Schedule расписания (tax, equity, leases, WC, interest)
- Corkscrew детализация (lease finance/operating, tax, WC)
- Debt instruments + cashflows
- PPE components, intangible assets
- Provisions, associates
- Macro factors, operational drivers
- Segments (financial + operational)
- Balancing adjustments

In [ ]:
sys.path.insert(0, str(ROOT / 'tools'))
from load_unified_excel import UnifiedExcelLoader
from engine import DB_PATH

loader = UnifiedExcelLoader(
    company_id=COMPANY,
    db_path=str(DB_PATH),
    dry_run=False,      # True для предпросмотра без записи
)
stats = loader.load(excel_path)

## Этап 3: Schedule Sheets (опционально)

Если unified loader не покрыл все schedule-листы, можно загрузить отдельно.
Поддерживает legacy имена листов (Intangibles_Schedule, Tax_Schedule и т.д.).

In [ ]:
# Опционально: загрузка schedule sheets отдельно
# Раскомментируйте, если нужно загрузить из другого Excel файла

# from load_schedule_sheets import PERIOD_HANDLERS, DIRECT_HANDLERS, get_period_map
# import sqlite3, pandas as pd
#
# schedule_excel = excel_path  # или другой файл
# conn = sqlite3.connect(str(DB_PATH))
# period_map = get_period_map(conn, COMPANY)
# xl = pd.ExcelFile(schedule_excel)
#
# for sheet, handler in PERIOD_HANDLERS.items():
#     if sheet in xl.sheet_names:
#         df = pd.read_excel(schedule_excel, sheet_name=sheet).dropna(how='all')
#         n = handler(conn, df, COMPANY, period_map, dry_run=False)
#         print(f'  {sheet}: {n} rows')
#
# conn.commit()
# conn.close()

## Verification

In [ ]:
import sqlite3
from engine import DB_PATH

conn = sqlite3.connect(str(DB_PATH))

# ── Core statements ──────────────────────────────────
print(f'=== {COMPANY}: History Data ===')
for stmt in ['is', 'bs', 'cf']:
    r = conn.execute(f'''
        SELECT COUNT(*) n, MIN(p.year) y0, MAX(p.year) y1
        FROM history_{stmt} h JOIN periods p ON h.period_id=p.period_id
        WHERE h.company_id=?
    ''', (COMPANY,)).fetchone()
    print(f'  {stmt.upper()}: {r[0]:>4} rows ({r[1]}-{r[2]})')

# ── Schedules & detail tables ────────────────────────
print(f'\n=== Schedule Tables ===')
schedule_tables = [
    ('debt_instruments', 'company_id'),
    ('debt_schedule', 'company_id'),
    ('tax_schedule', 'company_id'),
    ('equity_schedule', 'company_id'),
    ('lease_schedule', 'company_id'),
    ('ppe_components', 'company_id'),
    ('intangible_assets', 'company_id'),
    ('sched_wc_corkscrew', 'company_id'),
    ('interest_paid_split', 'company_id'),
    ('sched_tax_corkscrew', 'company_id'),
    ('provisions_schedule', 'company_id'),
    ('associates_schedule', 'company_id'),
    ('segment_data', 'company_id'),
    ('macro_factors', 'company_id'),
    ('operational_drivers', 'company_id'),
    ('balancing_adjustments', 'company_id'),
]
for table, col in schedule_tables:
    try:
        n = conn.execute(f'SELECT COUNT(*) FROM {table} WHERE {col}=?', (COMPANY,)).fetchone()[0]
        if n > 0:
            print(f'  {table:30s} {n:>5} rows')
    except Exception:
        pass

# ── BS identity check ────────────────────────────────
print(f'\n=== BS Identity (last 3 years) ===')
years = [r[0] for r in conn.execute('''
    SELECT DISTINCT p.year FROM history_bs h
    JOIN periods p ON h.period_id=p.period_id
    WHERE h.company_id=? ORDER BY p.year DESC LIMIT 3
''', (COMPANY,)).fetchall()]
for yr in sorted(years):
    metrics = {}
    for r in conn.execute('''
        SELECT h.metric, h.value FROM history_bs h
        JOIN periods p ON h.period_id=p.period_id
        WHERE h.company_id=? AND p.year=?
    ''', (COMPANY, yr)).fetchall():
        metrics[r[0]] = r[1]
    ta = metrics.get('total_assets', 0)
    tl = metrics.get('total_liabilities', 0)
    te = metrics.get('total_equity', 0)
    nci = metrics.get('nci', 0)
    diff = ta - tl - te - nci
    ok = '✓' if abs(diff) < 1e3 else '✗'
    print(f'  {yr}: A=${ta/1e6:,.0f}M  L+E=${(tl+te+nci)/1e6:,.0f}M  Δ=${diff/1e6:+,.2f}M {ok}')

conn.close()
print('\n✅ Data loading complete')